### JSON & JSONL Handling

## JSON : JSON stands for JavaScript Object Notation. It is the universal format for exchanging data between systems — especially APIs.
- Example of a JSON :
{
    "model": "gpt-4",
    "score": 0.91,
    "passed": true,
    "tags": ["factual", "grounded"]
}

 JSON maps directly to Python dicts and lists:
-  ![JSON to Python dicts](json_to_dicts.png)

Few things I will use constantly :
1. json.loads() - String to Dict : 
2. json.dumps() - Dict to String : You have a Python dict. Convert it to a JSON string to send to an API or save somewhere.
3. json.load() — Read JSON File : Read a .json file directly into a Python dict.
4. json.dump() - Write Dict to JSON File : Save a Python dict to a .json file.
5. Safe JSON Parsing — Always Use try/except : LLM outputs are unpredictable. Always wrap JSON parsing in error handling
try:
    data = json.loads(raw)
except json.JSONDecodeError:
    data = {}

(See examples below for each of above)

## JSONL : JSONL stands for JSON Lines. Each line in the file is a separate, complete JSON object.
- Example of a JSONL :
{"id": "case_1", "input": "What is 2+2?", "expected": "4"}
{"id": "case_2", "input": "Capital of France?", "expected": "Paris"}
{"id": "case_3", "input": "Who wrote Hamlet?", "expected": "Shakespeare"}

Why JSONL and not JSON for eval datasets?
- ![JSON to Python dicts](json_vs_jsonl.png)
- Industry standard : JSON for Config files, API responses and JSONL for Eval datasets, Logs and model outputs

Common things I will use constantly :
6. JSONL read : 
with open("data.jsonl") as f:
    rows = [json.loads(line.strip()) for line in f]

7. JSONL write
with open("data.jsonl", "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

8. JSONL append
with open("data.jsonl", "a") as f:
    f.write(json.dumps(new_row) + "\n")     

(See examples below for each of above)  

## Common mistakes 
- Mistake 1 — Using json.load() on a string instead of json.loads()

- Mistake 2 — Forgetting strip() when reading JSONL

- Mistake 3 — Opening file in wrong mode
    open("results.jsonl", "r")   # read only
    open("results.jsonl", "w")   # write — overwrites everything
    open("results.jsonl", "a")   # append — adds to end

## Real AI Eval Pipeline — Full Pattern 
See code below to understand full flow

In [1]:
# JSON Handling

## 1. json.loads() — String to Dict
import json

# Raw string you get from an API response
raw = '{"model": "gpt-4", "score": 0.91, "passed": true}'

# Convert to Python dict
data = json.loads(raw)

print(data["model"])    # "gpt-4"
print(data["score"])    # 0.91
print(data["passed"])   # True  ← JSON true becomes Python True

#-----------------------------------------------
## 2. json.dumps() — Dict to String

result = {
    "model": "gpt-4",
    "score": 0.91,
    "passed": True,
    "tags": ["factual", "grounded"]
}

# Convert to JSON string
raw = json.dumps(result)
print(raw)
# '{"model": "gpt-4", "score": 0.91, "passed": true, "tags": ["factual", "grounded"]}'

# Pretty print — indent=2 makes it readable
pretty = json.dumps(result, indent=2)
print(pretty)
# {
#   "model": "gpt-4",
#   "score": 0.91,
#   "passed": true,
#   "tags": [
#     "factual",
#     "grounded"
#   ]
# }

#-----------------------------------------------
## 3. json.load() — Read JSON File

with open("config.json", "r") as f:
    config = json.load(f)

print(config["model"])        # "gpt-4"
print(config["temperature"])  # 0.7

# Your config.json file looks like:
{
    "model": "gpt-4",
    "temperature": 0.7,
    "max_tokens": 512
}

#-----------------------------------------------
## 4. json.dump() — Write Dict to JSON File

results = {
    "run_id": "eval_001",
    "total": 100,
    "passed": 87,
    "failed": 13
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

# results.json now contains:
# {
#   "run_id": "eval_001",
#   "total": 100,
#   "passed": 87,
#   "failed": 13
# }

#-----------------------------------------------
## 5. Safe JSON Parsing — Always Use try/except

def parse_response(raw: str) -> dict:
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        return {}

# Works fine
parse_response('{"score": 0.9}')       # {"score": 0.9}

# Broken JSON — doesn't crash your pipeline
parse_response('not valid json')        # {}  ← empty dict, safe fallback
parse_response('{"score": 0.9')        # {}  ← missing closing brace, safe

# This pattern is used in every production eval pipeline. LLMs sometimes return 
# malformed JSON — your harness must handle it gracefully.

#-----------------------------------------------
## 6. Reading a JSONL File

cases = []
with open("eval_cases.jsonl", "r") as f:
    for line in f:
        case = json.loads(line.strip())   # parse each line as JSON
        cases.append(case)

print(cases)
# [
#   {"id": "case_1", "input": "What is 2+2?", "expected": "4"},
#   {"id": "case_2", "input": "Capital of France?", "expected": "Paris"},
# ]

#-----------------------------------------------
## 7. Writing a JSONL File

results = [
    {"id": "case_1", "score": 0.9,  "passed": True},
    {"id": "case_2", "score": 0.4,  "passed": False},
    {"id": "case_3", "score": 0.85, "passed": True},
]

with open("results.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")   # write each dict as one line

#-----------------------------------------------
## 8. Appending to a JSONL File

new_result = {"id": "case_4", "score": 0.75, "passed": False}

with open("results.jsonl", "a") as f:    # "a" = append mode
    f.write(json.dumps(new_result) + "\n")

#-----------------------------------------------
## Real AI Eval Pipeline — Full Pattern

def run_eval(input_path: str, output_path: str):

    # ─────────────────────────────────────────
    # STEP 1 — READ test cases from input file
    # ─────────────────────────────────────────
    cases = []                                      # empty list to collect cases
    with open(input_path, "r") as f:                # open eval_cases.jsonl for reading
        for line in f:                              # loop through file line by line
            case = json.loads(line.strip())         # convert each line string → dict
            cases.append(case)                      # add dict to cases list

    # cases is now a list of dicts in memory:
    # [
    #   {"id": "case_1", "input": "What is 2+2?", "expected": "4"},
    #   {"id": "case_2", "input": "Capital of France?", "expected": "Paris"},
    # ]

    # ─────────────────────────────────────────
    # STEP 2 — PROCESS each case through eval
    # ─────────────────────────────────────────
    results = []                                    # empty list to collect results
    for case in cases:                              # loop through each test case
        score  = evaluate(                          # run your eval logic — returns 0.0 to 1.0
                    case["input"],                  # the question/prompt
                    case["expected"]                # the expected answer
                 )
        passed = score >= 0.8                       # True or False
        results.append({                            # build result dict
            "id":     case["id"],                   # carry forward the case id
            "score":  score,                        # the score evaluate() returned
            "passed": passed                        # True or False
        })

    # results is now a list of dicts in memory:
    # [
    #   {"id": "case_1", "score": 0.9,  "passed": True},
    #   {"id": "case_2", "score": 0.4,  "passed": False},
    # ]

    # ─────────────────────────────────────────
    # STEP 3 — WRITE results to output file
    # ─────────────────────────────────────────
    with open(output_path, "w") as f:               # open results.jsonl for writing
        for result in results:                      # loop through each result dict
            f.write(json.dumps(result) + "\n")      # convert dict → string + newline → write to file

    # results.jsonl on disk now contains:
    # {"id": "case_1", "score": 0.9, "passed": true}
    # {"id": "case_2", "score": 0.4, "passed": false}

    # ─────────────────────────────────────────
    # STEP 4 — PRINT summary to console
    # ─────────────────────────────────────────
    total  = len(results)                           # total number of cases
    passed = sum(1 for r in results if r["passed"]) # count only passed ones
    print(f"Passed {passed}/{total} — {passed/total*100:.1f}%")
    # Passed 1/2 — 50.0%


# ─────────────────────────────────────────
# HOW TO CALL IT
# ─────────────────────────────────────────
run_eval(
    input_path  = "eval_cases.jsonl",   # file to read test cases from
    output_path = "results.jsonl"       # file to write results to
)


gpt-4
0.91
True
{"model": "gpt-4", "score": 0.91, "passed": true, "tags": ["factual", "grounded"]}
{
  "model": "gpt-4",
  "score": 0.91,
  "passed": true,
  "tags": [
    "factual",
    "grounded"
  ]
}


FileNotFoundError: [Errno 2] No such file or directory: 'config.json'

In [41]:
### REAL WORLD AI JSON TEST (8 Questions)

## Q1 — Safe parsing
# You are building an eval harness. Your LLM sometimes returns valid JSON, sometimes returns garbage. Write a function safe_parse(raw: str) -> dict that:
# - Returns the parsed dict if JSON is valid
# - Returns {"error": "invalid json"} if parsing fails

# Test it against both:
# - safe_parse('{"model": "gpt-4", "score": 0.9}')   # should work
# - safe_parse('not valid json at all')    # should not crash

import json

def safe_parse(raw: str) -> dict :
    try :
        return json.loads(raw)
    except json.JSONDecodeError as e :
        return {"error": "invalid json"}    

print(safe_parse('{"model": "gpt-4", "score": 0.9}')) # {'model': 'gpt-4', 'score': 0.9}
safe_parse('not valid json at all')  # {'error': 'invalid json'}

# --------------------------------------------------------------------
## Q2 - JSONL reading
# You have this eval_dataset.jsonl file on disk (check data_structures/lab_notebook/eval_dataset.jsonl). 
# Write code to:
# - Read all cases into a list called cases
# - Print the total number of cases loaded
# - Print the prompt of the first case

cases = []

with open("eval_dataset.jsonl", "r" ) as f:
    for line in f:
        case = json.loads(line.strip()) # converts from str to dict
        cases.append(case)

print(cases) # Read all cases into a list called cases
print(len(cases)) # Print the total number of cases loaded
print(cases[0]["prompt"]) # What is ML?

# --------------------------------------------------------------------
## Q3 - JSONL writing
# You just finished an eval run. You have this results list in memory:
pythonresults = [
    {"id": "q1", "score": 0.95, "passed": True},
    {"id": "q2", "score": 0.4,  "passed": False},
    {"id": "q3", "score": 0.88, "passed": True},
]
# Write code to save this to eval_results.jsonl — one result per line.

with open("eval_results.jsonl", "w") as f:
    for result in pythonresults:
        f.write(json.dumps(result) + "\n") # Now check the empty 'eval_results.jsonl' file and it should have 3 json ojects added.

# --------------------------------------------------------------------
## Q4 - JSON config file
# You have this config.json file - AI_Learnings/Python/data_structures/lab_notebook/config.json
# Write code to:
# - Load the config
# - Print the model name
# - Update temperature to 0.2
# - Save the updated config back to the same file

with open("config.json","r") as f:
    dict_config = json.load(f)
    print(dict_config) # {'model': 'gpt-4', 'temperature': 0.7, 'max_tokens': 512, 'pass_threshold': 0.8}

    print(dict_config["model"]) # gpt-4
    dict_config["temperature"] = 0.2
    print(dict_config) # {'model': 'gpt-4', 'temperature': 0.2, 'max_tokens': 512, 'pass_threshold': 0.8}

with open("config.json","w") as f:
    json.dump(dict_config, f, indent=2) 

# --------------------------------------------------------------------
## Q5 — Combining JSON + list comprehension
# You have a JSONL results file : model_score.jsonl. Read it and use a single list comprehension to extract 
# only the id values of failed cases (where passed is False).
# Expected O/P : ["q2", "q4"]

failed_ids = []

with open("model_score.jsonl", "r") as f:
  rows = [json.loads(row) for row in f] # from jsonl to dict using list comprehension

  failed_ids = [r["id"] for r in rows if not r["passed"]]

print(failed_ids) #['q2', 'q4']

# --------------------------------------------------------------------
## Q6 — Appending to JSONL
# Your eval pipeline ran 3 cases earlier and saved results to 'eval_results.jsonl'. 
# Now a 4th case just finished:
pythonnew_result = {"id": "q4", "score": 0.72, "passed": False}
# Write code to append this result to the existing file without overwriting the previous 3 results.

with open("eval_results.jsonl", "a") as f:
    f.write(json.dumps(pythonnew_result) + "\n") # Check o/p in 'eval_results.jsonl'

# --------------------------------------------------------------------
## Q7 — json.dumps() with indent
# You just finished an eval run and want to save a human readable summary report that your team can open and read easily in any text editor.
summary = {
    "run_id": "eval_run_003",
    "model": "gpt-4",
    "total_cases": 100,
    "passed": 87,
    "failed": 13,
    "pass_rate": "87%"
}
# Write code to save this to summary.json with clean indentation of 2 spaces.

with open("summary_report.json", "w") as f:
    json.dump(summary, f, indent=2)

# --------------------------------------------------------------------
# Q8 — json.load() vs json.loads() — spot the bug
# Your colleague wrote this code to parse an LLM API response. It crashes every time. 
# Find the bug and fix it. Explain in one line why it was wrong.

import json

# Raw response string coming back from LLM API
api_response = '{"model": "claude-3", "score": 0.91, "passed": true}'

# Parse it
data = json.loads(api_response) # This had the bug : json.load(api_response). Changed it to correct one.
print(data["model"])





{'model': 'gpt-4', 'score': 0.9}
[{'id': 'q1', 'prompt': 'What is ML?', 'expected': 'Machine Learning'}, {'id': 'q2', 'prompt': 'What is AI?', 'expected': 'Artificial Intelligence'}, {'id': 'q3', 'prompt': 'What is NLP?', 'expected': 'Natural Language Processing'}]
3
What is ML?
{'model': 'gpt-4', 'temperature': 0.2, 'max_tokens': 512, 'pass_threshold': 0.8}
gpt-4
{'model': 'gpt-4', 'temperature': 0.2, 'max_tokens': 512, 'pass_threshold': 0.8}
['q2', 'q4']
claude-3
